# Future Interns – Task 2: Support Ticket Classification

Build an ML system to classify customer support tickets and assign priority levels.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, ConfusionMatrixDisplay
import joblib

df = pd.read_csv("support_tickets.csv")
df.head()

In [ ]:
print(df.isnull().sum())
print(df["category"].value_counts())
print(df["priority"].value_counts())

In [ ]:
X = df["ticket_text"]
yc = df["category"]
yp = df["priority"]

X_train, X_test, yc_train, yc_test, yp_train, yp_test = train_test_split(
    X, yc, yp, test_size=0.2, random_state=42, stratify=yc
)

category_model = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1,2))),
    ("clf", LogisticRegression(max_iter=1000))
])
priority_model = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1,2))),
    ("clf", LogisticRegression(max_iter=1000))
])

category_model.fit(X_train, yc_train)
priority_model.fit(X_train, yp_train)

In [ ]:
cat_pred = category_model.predict(X_test)
pri_pred = priority_model.predict(X_test)

print("CATEGORY ACCURACY:", accuracy_score(yc_test, cat_pred))
print(classification_report(yc_test, cat_pred))

print("PRIORITY ACCURACY:", accuracy_score(yp_test, pri_pred))
print(classification_report(yp_test, pri_pred))

In [ ]:
ConfusionMatrixDisplay.from_predictions(yc_test, cat_pred, xticks_rotation=45)
plt.title("Ticket Category Confusion Matrix")
plt.tight_layout()
plt.show()

ConfusionMatrixDisplay.from_predictions(yp_test, pri_pred)
plt.title("Priority Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
def classify_ticket(text):
    return category_model.predict([text])[0], priority_model.predict([text])[0]

examples = [
    "My payment was charged twice and I need urgent help",
    "The mobile application crashes when I open it",
    "Can I change my delivery date?"
]

for text in examples:
    category, priority = classify_ticket(text)
    print("Ticket:", text)
    print("Category:", category)
    print("Priority:", priority)
    print()

In [ ]:
joblib.dump(category_model, "ticket_category_model.pkl")
joblib.dump(priority_model, "ticket_priority_model.pkl")
print("Models saved.")